In [1]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
# pip install --force-reinstall "transformers>=4.48.0,<5.0.0"

In [3]:
pip install --upgrade unsloth unsloth-zoo transformers huggingface_hub

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 14.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 32.2 MB/s eta 0:00:0000:01m
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.3
    Uninstalling hf-xet-1.4.3:
      Successfully uninstalled hf-xet-1.4.3
  Attempting uninstall: click
    Found existing installation: click 8.3.2
    Uninstalling click-8.3.2:
      Successfully uninstalled click-8.3.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.2
    Uninstalling transformers-4.56.2:


In [4]:
!pip install -U huggingface_hub transformers datasets

  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 89.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.2 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.6.9 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth 2026.6.9 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.

In [5]:
import huggingface_hub

print(huggingface_hub.__version__)

1.21.0


In [6]:
import unsloth

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
%%capture
!pip install evaluate json_repair

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from unsloth import FastLanguageModel
from datasets import Dataset
import evaluate
import numpy as np
import json

In [9]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [10]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [11]:
BATCH_SIZE = 50  # tune this based on GPU speed + timeout risk
repo_id = "dinushiTJ/nz_research_commons_inference_results"

In [12]:
from datasets import load_dataset

ds = load_dataset(repo_id, split="train")

README.md:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3318 [00:00<?, ? examples/s]

In [13]:
ds

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'BaseEmbGemZeroLabel', 'BaseEmbGemZeroID', 'BaseEmbGemZeroScore', 'Base4BResponse', 'FTEmbGemZeroLabel', 'FTEmbGemZeroID', 'FTEmbGemZeroScore', 'FT4BResponse'],
    num_rows: 3318
})

In [14]:
ds[0]['prompt']

'You are a specialist in classifying academic research papers by their relevance to Māori topics. Classify the academic research paper below into EXACTLY ONE category:\n- maori_origin: Substantively connected to Māori people, communities, land, resources, or knowledge\n- non_maori_origin: No meaningful Māori connection\n\nQUALIFIES AS maori_origin if it substantively involves ANY of:\n\n1. PEOPLE & COMMUNITY\n   • Māori people, whānau, hapū, or iwi as primary subject or participant group\n   • Māori health, education, economic, justice, or social outcomes\n   • Māori demographic studies where Māori outcomes are the primary focus\n\n2. CULTURE, IDENTITY & LANGUAGE\n   • Te reo Māori — acquisition, revitalisation, or documentation\n   • Māori cultural practices, tikanga, identity, whakapapa, or arts\n   • Māori oral traditions, media, or digital representation\n\n3. HISTORY, POLITICS & LAW\n   • Treaty of Waitangi, Māori land rights, Waitangi Tribunal, Treaty settlements\n   • Māori sove

In [15]:
from unsloth import FastModel
import torch

ft_model, ft_tokenizer = FastModel.from_pretrained(
    model_name = "dinushiTJ/gemma-3-4b-research-commons-lora", # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = 4096,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.6.9: Fast Gemma3 patching. Transformers: 5.12.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

adapter_model.safetensors:   0%|          | 0.00/59.7M [00:00<?, ?B/s]

In [16]:
FastLanguageModel.for_inference(ft_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (embeddings): SiglipVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
            (position_embedding): Embedding(4096, 1152)
          )
          (encoder): SiglipEncoder(
            (layers): ModuleList(
              (0-26): 27 x SiglipEncoderLayer(
                (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                (self_attn): SiglipAttention(
                  (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                  (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
                )
 

In [17]:
from time import perf_counter
import json
from json_repair import repair_json


def generate_ft_response(user_message: str):
    start_time = perf_counter()

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": user_message,
                }
            ],
        }
    ]

    text = ft_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = ft_tokenizer(
        text=[text],
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )

    end_time = perf_counter()

    raw_output = ft_tokenizer.decode(
        outputs[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
    )

    # Safety: normalize output
    if isinstance(raw_output, list):
        raw_output = "".join(raw_output)

    parsed_output = None
    repaired = False

    # ---- JSON repair + parsing ----
    try:
        repaired_json = repair_json(raw_output)

        if isinstance(repaired_json, str):
            parsed_output = json.loads(repaired_json)
        else:
            parsed_output = repaired_json

        repaired = True

    except Exception as e:
        print("[WARNING]: JSON repair failed, returning raw output")
        print("[ERROR]:", repr(e))
        print("Raw output preview:", raw_output[:500])

        parsed_output = raw_output  # fallback

    return {
        "output": parsed_output,
        "raw_output": raw_output,
        "time": end_time - start_time,
        "json_ok": repaired,
    }


In [18]:
user_msg = ds[0]["prompt"]

In [19]:
generate_ft_response(user_msg)

{'output': {'label': 'non_maori_origin',
  'confidence': 1.0,
  'reason': 'The research is conducted in Papua New Guinea, focusing on female education transition within a Papua New Guinean cultural context and the National Gender Equity Policy. There is no mention of Māori people, culture, or New Zealand context.'},
 'raw_output': '{"label": "non_maori_origin", "confidence": 1.0, "reason": "The research is conducted in Papua New Guinea, focusing on female education transition within a Papua New Guinean cultural context and the National Gender Equity Policy. There is no mention of Māori people, culture, or New Zealand context."}',
 'time': 143.116416502,
 'json_ok': True}

In [20]:
def generate_ft_response_batch(user_messages):

    messages_batch = []
    for msg in user_messages:
        messages_batch.append([
            {
                "role": "user",
                "content": [{"type": "text", "text": msg}],
            }
        ])

    texts = [
        ft_tokenizer.apply_chat_template(
            m,
            add_generation_prompt=True,
            tokenize=False,
        )
        for m in messages_batch
    ]

    inputs = ft_tokenizer(
        text=texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        add_special_tokens=False,
    ).to("cuda")

    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )

    decoded = ft_tokenizer.batch_decode(
        outputs[:, inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
    )

    return decoded

In [21]:
# def run_ft_response(example):
#     try:
#         prompt = example["prompt"]
#         response = generate_ft_response(prompt)

#         # Store as string (JSON-safe)
#         if isinstance(response, dict):
#             response = json.dumps(response, ensure_ascii=False)

#         return {"FT4BResponse": response}

#     except Exception as e:
#         print(e)
#         return {"FT4BResponse": "__ERR__"}


In [22]:
# test_ds = ds.select(range(2))
# test_ds = test_ds.map(run_ft_response)

In [23]:
# err_count = test_ds["FT4BResponse"].count("__ERR__")
# print("ERR count:", err_count)

In [24]:
# print(test_ds[0]["FT4BResponse"])
# print(test_ds[1]["FT4BResponse"])

In [25]:
if "FT4BResponse" not in ds.column_names:
    ds = ds.add_column(
        "FT4BResponse",
        ["__ERR__"] * len(ds)
    )
    print("Added column")

In [26]:
repo_id = "dinushiTJ/nz_research_commons_inference_results"
BATCH_SIZE = 4
RETRY_ALL = False

In [27]:
def should_run(example):
    if RETRY_ALL:
        return True

    v = example.get("FT4BResponse")

    return v in ["__ERR__"]


In [28]:
remaining_indices = [
    i for i, ex in enumerate(ds)
    if should_run(ex)
]

print(f"Remaining: {len(remaining_indices)}")


Remaining: 886


In [29]:
from tqdm.auto import tqdm
import json
import time
from huggingface_hub.utils import HfHubHTTPError

BATCH_SIZE = 4
PUSH_EVERY = 100   # push to HF every 100 completed records, not every batch

repo_id = "dinushiTJ/nz_research_commons_inference_results"

# Make sure column exists
if "FT4BResponse" not in ds.column_names:
    ds = ds.add_column("FT4BResponse", ["__ERR__"] * len(ds))

def should_run(example):
    v = example.get("FT4BResponse")
    return v in ["__ERR__", None, ""]

remaining_indices = [
    i for i, ex in enumerate(ds)
    if should_run(ex)
]

print("Remaining:", len(remaining_indices))

col = list(ds["FT4BResponse"])

pbar = tqdm(
    total=len(remaining_indices),
    desc="Inference"
)

completed_since_push = 0

for batch_start in range(0, len(remaining_indices), BATCH_SIZE):

    batch = remaining_indices[batch_start: batch_start + BATCH_SIZE]
    prompts = [ds[i]["prompt"] for i in batch]

    try:
        outputs = generate_ft_response_batch(prompts)

    except Exception as e:
        print("Batch failed:", e)
        outputs = ["__ERR__"] * len(batch)

    for idx, out in zip(batch, outputs):

        try:
            # Store raw model output as string
            # Evaluation function can parse this later
            if isinstance(out, dict):
                col[idx] = json.dumps(out, ensure_ascii=False)
            else:
                col[idx] = str(out)

        except Exception as e:
            print(f"Row {idx} failed:", e)
            col[idx] = "__ERR__"

        completed_since_push += 1
        pbar.update(1)

    # Update dataset in memory
    ds = ds.remove_columns(["FT4BResponse"])
    ds = ds.add_column("FT4BResponse", col)

    completed_total = len([x for x in col if x not in ["__ERR__", None, ""]])

    # Push only every 100 completed records
    if completed_since_push >= PUSH_EVERY:
        try:
            ds.push_to_hub(repo_id, token=HF_TOKEN)
            print(f"Pushed checkpoint to HF. Completed rows: {completed_total}")
            completed_since_push = 0

        except HfHubHTTPError as e:
            print("HF push failed, probably rate limit.")
            print(e)
            print("Continuing inference in memory. Will try pushing later.")

# Final push
ds.push_to_hub(repo_id, token=HF_TOKEN)

print("Done")

Remaining: 886


Inference:   0%|          | 0/886 [00:00<?, ?it/s]

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
[datasets.arrow_dataset|WARNING]Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done


In [30]:
err_count = sum(1 for x in ds["FT4BResponse"] if x == "__ERR__")
print("ERR count:", err_count)

ERR count: 4


In [31]:
import pandas as pd

col = ds["FT4BResponse"]

# convert to pandas for easy analysis
s = pd.Series(col)

# normalize null-like values
null_count = s.isna().sum() + (s == "").sum()

unique_labels = s.dropna()
unique_labels = unique_labels[unique_labels != ""].unique()

print("Null / empty count:", null_count)
print("Num unique labels:", len(unique_labels))

Null / empty count: 0
Num unique labels: 3315


In [32]:
from datasets import DatasetDict

fixed_ds = DatasetDict({
    "train": ds
})

fixed_ds.push_to_hub(repo_id)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
[datasets.arrow_dataset|WARNING]Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/dinushiTJ/nz_research_commons_inference_results/commit/7816ba5c8809babcceeb5182533e63b473fa4e28', commit_message='Upload dataset', commit_description='', oid='7816ba5c8809babcceeb5182533e63b473fa4e28', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/dinushiTJ/nz_research_commons_inference_results', endpoint='https://huggingface.co', repo_type='dataset', repo_id='dinushiTJ/nz_research_commons_inference_results'), pr_revision=None, pr_num=None)

End